# ByteMe: Tokenizer Comparison

Compare 4 tokenization strategies on the same text:
- Byte-level BPE (GPT-4 style)
- BPE (standard)
- WordPiece (BERT)
- Unigram (T5/XLNet)

We test on regular text, code, emojis, multilingual, and URLs.

In [ ]:
import os
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import AutoTokenizer
import sentencepiece as spm
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

## Test samples

We need text that covers different scenarios. Regular english, spanish, code, emojis, URLs, and mixed content. Each tokenizer will handle these differently.

In [ ]:
TEST_SAMPLES = {
    "english": "The quick brown fox jumps over the lazy dog. Natural language processing has transformed how we interact with computers.",
    "spanish": "El procesamiento de lenguaje natural ha transformado la forma en que interactuamos con las computadoras. Los modelos de lenguaje son cada vez mas potentes.",
    "code": "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)\n\nresult = [fibonacci(i) for i in range(10)]",
    "emojis": "I love this movie! 🎬🍿 The acting was superb 👏👏👏 10/10 would recommend 🔥💯",
    "urls": "Check out https://huggingface.co/docs/transformers/main/tokenizer_summary and also visit http://arxiv.org/abs/1706.03762 for the original paper.",
    "mixed": "El modelo GPT-4 usa byte-level BPE con vocab de ~100k tokens. Mas info en https://openai.com 🤖",
    "rare_words": "The pneumonoultramicroscopicsilicovolcanoconiosis patient visited the otorhinolaryngologist for a consultation.",
    "numbers": "The model has 175,000,000,000 parameters and was trained on 300B tokens with a learning rate of 3e-4.",
}

print(f"Created {len(TEST_SAMPLES)} test samples")
for name, text in TEST_SAMPLES.items():
    print(f"  {name}: {len(text)} chars")

## Load tokenizers

We load one real tokenizer for each strategy. These are all pretrained and available from HuggingFace.

- **Byte-level BPE**: GPT-2 tokenizer (same family as GPT-4, just smaller vocab)
- **BPE**: GPT-NeoX / Pythia tokenizer
- **WordPiece**: BERT base uncased
- **Unigram**: T5 small (uses SentencePiece under the hood)

In [ ]:
TOKENIZERS = {
    "byte_bpe": {
        "name": "Byte-level BPE",
        "model": "gpt2",
        "example": "GPT-2, GPT-4, Llama",
    },
    "bpe": {
        "name": "BPE",
        "model": "EleutherAI/pythia-70m",
        "example": "Pythia, GPT-NeoX",
    },
    "wordpiece": {
        "name": "WordPiece",
        "model": "bert-base-uncased",
        "example": "BERT, DistilBERT",
    },
    "unigram": {
        "name": "Unigram",
        "model": "t5-small",
        "example": "T5, XLNet, ALBERT",
    },
}

loaded = {}
for key, info in TOKENIZERS.items():
    print(f"Loading {info['name']} ({info['model']})...")
    tok = AutoTokenizer.from_pretrained(info["model"])
    loaded[key] = tok
    print(f"  Vocab size: {tok.vocab_size:,}")

print(f"\nLoaded {len(loaded)} tokenizers")